# 06 — Downstream RAG and RGB Robustness

Final answer evaluation is downstream of the common MIGRES filtering/evidence/state pipeline. Accuracy is defined here as normalized exact-answer correctness with gold aliases; it is not an LLM judge and not a contains-match heuristic. The RGB suite is intentionally downscoped to downstream robustness on its own prepared cases; RGB passages are not silently mixed into the main retrieval corpus. Negative rejection is therefore measured at output level, while the retrievers themselves remain fixed Top-K systems.


In [ ]:
from pathlib import Path
import sys, json, yaml, platform
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
print("Project root:", ROOT)
try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("Selected device:", DEVICE)
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    DEVICE = "cpu"
    print("Torch/GPU probe unavailable:", type(e).__name__, e)
PLAN = yaml.safe_load((ROOT / "configs" / "experiments.yaml").read_text())
BASE_CONFIG = [ROOT / "configs" / "default.yaml", ROOT / "configs" / "gemini.yaml"]


In [ ]:
RUN_EXPERIMENT=False
LIMIT_PER_DATASET=None
METHODS=['sar']+PLAN['baselines']['systems']
OUT_ROOT=ROOT/'outputs/experiments/06_downstream_rag'
RGB_CASES_PATH=None
RGB_PREDICTIONS_PATH=None


In [ ]:
from sar_migres.utils.config import load_config
from sar_migres.datasets.loader import load_phase_questions,dataset_statistics
from sar_migres.datasets.relevance import load_rgb
from sar_migres.evaluation.rgb import RGBRobustnessEvaluation
from sar_migres.experiments.registry import baseline_overrides
from sar_migres.experiments.parameter_selection import frozen_setting
from sar_migres.experiments.iterative import answer_metrics_from_run
from sar_migres.experiments.runner import SARMigresPipeline
base=load_config(*BASE_CONFIG)
from sar_migres.utils.config import validate_canonical_protocol
validate_canonical_protocol(base)
fp=ROOT/'outputs/experiments/01_parameter_selection/selected_config.json'
if not RUN_EXPERIMENT:
    print('Preflight only. Final split:', base.data.final_split)
    print('Frozen parameters:', 'present' if fp.exists() else 'MISSING (required before RUN_EXPERIMENT=True)')
else:
    k,w=frozen_setting(fp)
    for ds in base.data.datasets:
        items=load_phase_questions(ds,base.data.dataset_paths[ds],'final',limit=LIMIT_PER_DATASET,final_split=base.data.final_split,validation_fraction=base.data.validation_fraction,validation_seed=base.data.validation_seed)
        print('Dataset:',ds,dataset_statistics(items))
        for method in METHODS:
            if method=='sar':
                ov={'retrieval':{'method':'sar'},'sar':{'k':k},'baseline':{'k':k}}
                if w: ov['sar'].update({'alpha':w.alpha,'beta':w.beta,'gamma':w.gamma,'weighting':'static'})
            else:
                ov=baseline_overrides(method,k=k,run_name=f'{ds}_{method}')
            ov['data']={'dataset':ds,'dataset_path':base.data.dataset_paths[ds],'split':base.data.final_split}
            ov['run']={'name':f'{ds}_{method}','output_dir':str(OUT_ROOT),'resume':False}
            pipe=SARMigresPipeline(load_config(*BASE_CONFIG,overrides=ov),run_id=f'{ds}_{method}').prepare()
            pipe.run_batch(items,resume=False)
            print(ds,method,answer_metrics_from_run(pipe.run_dir))

if RGB_CASES_PATH and RGB_PREDICTIONS_PATH:
    cases=RGBRobustnessEvaluation.from_records(load_rgb(RGB_CASES_PATH))
    preds=[json.loads(x) for x in Path(RGB_PREDICTIONS_PATH).read_text().splitlines() if x.strip()]
    print(json.dumps(cases.evaluate(preds),indent=2,allow_nan=True))
else:
    print('RGB retrieval integration is OFF by design:',PLAN['rgb']['note'])
